# Design arm · D1 — Event-level recall table

Post hoc arm (see README.md). Builds one row per **recall event** linked to a corpus device, carrying FDA's root cause and the fields reviewers need, and maps root causes to the five study categories with cause_map.py.

Network: none. Reads only frozen inputs from ../snapshot and ../data. Writes to design_arm/data/.

In [1]:
import gzip, json
from pathlib import Path
import pandas as pd
from cause_map import map_root_cause, FDA_ROOT_CAUSE_MAP, CATEGORIES, UNCLEAR

OUT = Path("data"); OUT.mkdir(exist_ok=True)
with gzip.open("../snapshot/recall_raw.json.gz", "rt") as f:
    recalls = json.load(f)
corpus = pd.read_csv("../data/corpus.csv", dtype=str)
VALID = set(corpus["k_number"])
edges = pd.read_csv("../data/predicate_edges.csv", dtype=str)
edges_r1 = pd.read_csv("../data/predicate_edges_rule1.csv", dtype=str)
CITED = set(edges["predicate_knumber"]) | set(edges_r1["predicate_knumber"])
print(f"{len(recalls)} recall product records loaded")

6297 recall product records loaded


### Device-level links (corpus K-number × recall product record)

In [2]:
link_rows, prod_rows = [], []
for rec in recalls:
    kns = sorted({str(k).strip().upper() for k in (rec.get("k_numbers") or [])} & VALID)
    if not kns:
        continue
    prod_rows.append({
        "res_event_number": rec.get("res_event_number"),
        "product_res_number": rec.get("product_res_number"),
        "event_date_initiated": rec.get("event_date_initiated"),
        "root_cause_description": rec.get("root_cause_description"),
        "reason_for_recall": rec.get("reason_for_recall"),
        "product_description": rec.get("product_description"),
        "code_info": rec.get("code_info"),
        "action": rec.get("action"),
        "recalling_firm": rec.get("recalling_firm"),
        "product_code": rec.get("product_code"),
    })
    for k in kns:
        link_rows.append({"k_number": k,
                          "res_event_number": rec.get("res_event_number"),
                          "product_res_number": rec.get("product_res_number"),
                          "event_date_initiated": rec.get("event_date_initiated")})
links = pd.DataFrame(link_rows).drop_duplicates()
prods = pd.DataFrame(prod_rows)
links.to_csv(OUT / "event_device_links.csv", index=False)
print(f"CHECKPOINT  device-recall links {len(links)} | corpus devices {links['k_number'].nunique()} | events {links['res_event_number'].nunique()}")

parent_recalled = set(pd.read_csv("../data/recalled_nodes.csv", dtype=str)["k_number"])
assert set(links["k_number"]) == parent_recalled, "recalled device set differs from parent recalled_nodes.csv"
print("CHECKPOINT  recalled device set matches parent pipeline")

CHECKPOINT  device-recall links 6852 | corpus devices 1691 | events 1698
CHECKPOINT  recalled device set matches parent pipeline


### Collapse to recall events and apply the FDA root-cause mapping

In [3]:
def join_unique(s):
    vals = [str(v).strip() for v in s.dropna() if str(v).strip()]
    return " || ".join(dict.fromkeys(vals))

events = (prods.groupby("res_event_number")
          .agg(event_date_initiated=("event_date_initiated", "min"),
               n_products=("product_res_number", "nunique"),
               root_cause_description=("root_cause_description", join_unique),
               reason_for_recall=("reason_for_recall", join_unique),
               product_description=("product_description", join_unique),
               code_info=("code_info", join_unique),
               action=("action", join_unique),
               recalling_firm=("recalling_firm", join_unique),
               product_codes=("product_code", join_unique))
          .reset_index())
ev_k = links.groupby("res_event_number")["k_number"].agg(lambda s: ";".join(sorted(set(s))))
events["k_numbers"] = events["res_event_number"].map(ev_k)
events["n_devices"] = events["k_numbers"].str.count(";") + 1
events["in_scope"] = events["k_numbers"].apply(lambda s: any(k in CITED for k in s.split(";")))

n_rc = prods.groupby("res_event_number")["root_cause_description"].nunique()
events["mixed_root_cause"] = events["res_event_number"].map(n_rc).fillna(0).gt(1)
events["fda_category"] = [UNCLEAR if m else map_root_cause(rc)
                          for m, rc in zip(events["mixed_root_cause"], events["root_cause_description"])]

unmapped = sorted({rc for rc in prods["root_cause_description"].dropna().unique()
                   if rc.strip() not in FDA_ROOT_CAUSE_MAP})
print("CHECKPOINT  FDA root causes not in cause_map (default to other_unclear):", unmapped or "none")
events.to_csv(OUT / "recall_events.csv", index=False)
print(f"CHECKPOINT  events {len(events)} | in scope (linked to a cited predicate) {int(events['in_scope'].sum())} | mixed root cause {int(events['mixed_root_cause'].sum())}")
print("In-scope events by FDA-mapped category:")
print(events[events["in_scope"]]["fda_category"].value_counts().reindex(CATEGORIES, fill_value=0).to_string())

CHECKPOINT  FDA root causes not in cause_map (default to other_unclear): none
CHECKPOINT  events 1698 | in scope (linked to a cited predicate) 1510 | mixed root cause 0
In-scope events by FDA-mapped category:
fda_category
design                   222
manufacturing_process    578
labeling                 240
packaging_sterility      133
other_unclear            337
